
Student Name: Sneha Gautam

Roll Number:A26MTAI0019

Practical Title: Repetition, Generators and Decorators

Date:25 October 2026

In [2]:

import time
import functools
import math


# 1. Decorator for recording execution time
def track_runtime(function):
    """Display the time taken by a function."""

    @functools.wraps(function)
    def timed_function(*args, **kwargs):
        start = time.perf_counter()
        output = function(*args, **kwargs)
        duration = time.perf_counter() - start

        print(
            f"[TIME] '{function.__name__}' completed in "
            f"{duration:.6f}s"
        )
        return output

    return timed_function


# 2. Generator to filter sensor values
@track_runtime
def filter_readings(
    data: list,
    minimum: float,
    /,
    *,
    upper_limit: float = 60.0
):
    """Generate numeric readings within the specified limits."""

    for item in data:
        if isinstance(item, bool):
            continue

        if not isinstance(item, (int, float)):
            continue

        if not math.isfinite(item):
            continue

        if minimum <= item <= upper_limit:
            yield item


if __name__ == "__main__":

    # 3. Process sensor data lazily
    sensor_data = [
        16.5, 24.0, "WARNING", 29.5,
        None, 35.0, 19.0, "NO SIGNAL", 42.5
    ]
    minimum_value = 20.0

    print("=== SENSOR DATA PROCESSING ===")

    valid_stream = filter_readings(sensor_data, minimum_value)

    converted_stream = (
        round((value * 9 / 5) + 32, 2)
        for value in valid_stream
    )

    for temperature in converted_stream:
        print(f"Temperature in Fahrenheit: {temperature:.2f} F")

    # 4. Check preserved function metadata
    assert filter_readings.__name__ == "filter_readings"
    assert filter_readings.__doc__ is not None

    print("Metadata verification successful.")

    # 5. Check positional-only parameters
    print("\n=== POSITIONAL ARGUMENT CHECK ===")

    try:
        filter_readings(
            data=sensor_data,
            minimum=minimum_value
        )
    except TypeError as error:
        print("Expected error:", error)

    # 6. Bonus: Test the upper limit
    print("\n=== UPPER LIMIT TEST ===")

    unusual_values = [999.9]

    print("Using the default limit:")
    for number in filter_readings(unusual_values, 20.0):
        print("Output:", number)

    print("Using a custom limit:")
    for number in filter_readings(
        unusual_values, 20.0, upper_limit=1000.0
    ):
        print("Output:", number)

    print("Trying a third positional argument:")
    try:
        filter_readings(unusual_values, 20.0, 1000.0)
    except TypeError as error:
        print("Expected error:", error)

    # 7. Check Boolean and non-finite values
    print("\n=== INVALID NUMERIC VALUES ===")

    unusual_data = [
        True, False, float("nan"),
        float("inf"), float("-inf"), 27.5
    ]

    for number in filter_readings(unusual_data, 0.0):
        print("Valid value:", number)

    # 8. Check that a generator can be consumed only once
    print("\n=== GENERATOR REUSE TEST ===")

    reading_stream = filter_readings([22.0, 26.0, 30.0], 20.0)

    print("First iteration:")
    for number in reading_stream:
        print(number)

    print("Second iteration:")
    for number in reading_stream:
        print(number)

    try:
        next(reading_stream)
    except StopIteration:
        print("The generator has no remaining values.")

    # 9. Empty output test
    print("\n=== EMPTY INPUT TEST ===")

    no_data = filter_readings(["ERROR", None, "OFFLINE"], 20.0)
    has_data = False

    for number in no_data:
        print(number)
        has_data = True

    if not has_data:
        print("The input contains no acceptable readings.")

    # 10. Measure time including generator consumption
    print("\n=== LARGE DATA TEST ===")

    many_readings = [28.0] * 1_000_000

    start = time.perf_counter()
    large_stream = filter_readings(many_readings, 20.0)

    total_readings = 0
    for number in large_stream:
        total_readings += 1

    elapsed = time.perf_counter() - start

    print("Total readings processed:", total_readings)
    print(f"Creation and consumption time: {elapsed:.6f}s")

    # 11. Demonstrate metadata without functools.wraps
    print("\n=== DECORATOR METADATA COMPARISON ===")

    def basic_decorator(function):
        """A decorator without metadata preservation."""

        def inner_function(*args, **kwargs):
            return function(*args, **kwargs)

        return inner_function

    @basic_decorator
    def demo_task():
        """A sample task used for comparison."""
        return "Completed"

    print("Displayed name:", demo_task.__name__)
    print("Displayed docstring:", demo_task.__doc__)

    print(
        "The wrapper metadata is displayed because "
        "functools.wraps was not applied."
    )


=== SENSOR DATA PROCESSING ===
[TIME] 'filter_readings' completed in 0.000004s
Temperature in Fahrenheit: 75.20 F
Temperature in Fahrenheit: 85.10 F
Temperature in Fahrenheit: 95.00 F
Temperature in Fahrenheit: 108.50 F
Metadata verification successful.

=== POSITIONAL ARGUMENT CHECK ===
Expected error: filter_readings() got some positional-only arguments passed as keyword arguments: 'data, minimum'

=== UPPER LIMIT TEST ===
Using the default limit:
[TIME] 'filter_readings' completed in 0.000002s
Using a custom limit:
[TIME] 'filter_readings' completed in 0.000002s
Output: 999.9
Trying a third positional argument:
Expected error: filter_readings() takes 2 positional arguments but 3 were given

=== INVALID NUMERIC VALUES ===
[TIME] 'filter_readings' completed in 0.000001s
Valid value: 27.5

=== GENERATOR REUSE TEST ===
[TIME] 'filter_readings' completed in 0.000001s
First iteration:
22.0
26.0
30.0
Second iteration:
The generator has no remaining values.

=== EMPTY INPUT TEST ===
[TIME] 

## Reflection: Generator Execution and Performance

During this practical, I learned how decorators and generators can be used to process sensor data efficiently in Python. The `track_runtime` decorator measures the time taken by a function using `time.perf_counter()`. I also understood that `functools.wraps` helps preserve the original function's name and documentation when a decorator is applied.

The most important concept I learned was lazy evaluation. A generator produces values one at a time instead of storing all the results in memory. In the sensor pipeline, readings are filtered first and then converted from Celsius to Fahrenheit as they are consumed by the loop. This approach is useful when processing large streams of data.

The timing experiment with one million readings helped me understand that measuring only generator creation does not represent the total processing time. Since generators perform their work during iteration, the time required to consume all the values must also be considered. I also tested invalid inputs, Boolean values, non-finite numbers, empty streams, and exhausted generators.

Finally, I learned the difference between positional-only and keyword-only parameters. These rules make function calls more predictable and allow an optional upper limit to be configured clearly. Overall, this practical improved my understanding of Python decorators, generator expressions, validation, and performance measurement.